# Cows Challenge: Keypoint Detection Quickstart

Este notebook executa o fluxo completo (Quickstart) com o subset do dataset para validação do pipeline de Pose Estimation da Ultralytics, utilizando apenas imagens reais validadas.

## 1. Inspeção do Dataset e Imagens Reais
Contagem das imagens reais em `data/raw_images/` e cruzamento com as anotações do Label Studio (`Key_points/`).

In [ ]:
import os
raw_imgs = len(os.listdir("../data/raw_images")) if os.path.exists("../data/raw_images") else 0
print(f"Total de imagens reais em data/raw_images: {raw_imgs}")

In [ ]:
!python ../src/inspect_dataset.py

from IPython.display import Markdown
try:
    display(Markdown("../outputs/reports/dataset_summary.md"))
except:
    print("Report missing.")

## 2. Validação das anotações

In [ ]:
!python ../src/validate.py
import json
try:
    with open("../outputs/reports/validation_report.json", "r") as f:
        val_data = json.load(f)
        print(f"Arquivos válidos: {val_data['valid_files']}")
        print(f"Arquivos com problema: {val_data['files_with_issues']}")
except:
    print("Validation report missing.")

## 3. Sanity Check (Ground Truth sobre imagens reais)
Visualizando 10 amostras reais com os esqueletos desenhados a partir das anotações originais.

In [ ]:
!python ../src/sanity_check.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

vis_dir = Path("../outputs/vis_samples")
if vis_dir.exists():
    imgs = list(vis_dir.glob("sanity_gt_*.jpg"))
    if imgs:
        # Show up to 5 images inline
        num_to_show = min(5, len(imgs))
        fig, axs = plt.subplots(1, num_to_show, figsize=(20, 5))
        if num_to_show == 1: axs = [axs]
        for ax, img_path in zip(axs, imgs[:num_to_show]):
            img = mpimg.imread(str(img_path))
            ax.imshow(img)
            ax.axis('off')
        plt.show()
    else:
        print("No sanity samples generated.")

## 4. Criação do Subset e Conversão YOLO Pose

In [ ]:
!python ../src/make_subset.py
# Gera o subset apenas com imagens reais encontradas, aplicando formato YOLO Pose

## 5. Treinamento Rápido no Subset (Ultralytics YOLO Pose)

In [ ]:
# Treinamento em 5 épocas com yolov8n-pose
!python ../src/train_pose.py --data ../data/subset_yolo_pose/data.yaml --epochs 5 --model yolov8n-pose.pt

## 6. Inferência e Avaliação

In [ ]:
!python ../src/evaluate_pose.py --data ../data/subset_yolo_pose/data.yaml
try:
    display(Markdown("../outputs/reports/summary.md"))
except:
    print("Evaluation Report missing.")

In [ ]:
!python ../src/visualize_predictions.py --data-dir ../data/subset_yolo_pose/images/val --num-samples 5

if vis_dir.exists():
    pred_imgs = list(vis_dir.glob("pred_*.jpg"))
    if pred_imgs:
        num_to_show = min(5, len(pred_imgs))
        fig, axs = plt.subplots(1, num_to_show, figsize=(20, 5))
        if num_to_show == 1:
            axs = [axs]
        for ax, img_path in zip(axs, pred_imgs[:num_to_show]):
            img = mpimg.imread(str(img_path))
            ax.imshow(img)
            ax.axis('off')
        plt.show()
    else:
        print("No prediction visualizations found.")